## импорты

In [1]:
import pandas as pd

## Загрузка данных

In [2]:
df = pd.read_csv('TextLanguage.csv')

# Определить необходимые поля и создать структуру набора данных

In [3]:
df.head(10)

,Text,Language
0,"Nature, in the broadest sense, is the natural...",English
1,"""Nature"" can refer to the phenomena of the phy...",English
2,"The study of nature is a large, if not the onl...",English
3,"Although humans are part of nature, human acti...",English
4,[1] The word nature is borrowed from the Old F...,English
5,"[2] In ancient philosophy, natura is mostly us...",English
6,"[3][4] \nThe concept of nature as a whole, the...",English
7,During the advent of modern scientific method ...,English
8,"[5][6] With the Industrial revolution, nature ...",English
9,"However, a vitalist vision of nature, closer t...",English


Как мы видим тут только два поля и исключение какого либо лишает смысла дальнейшую работу

# Провести предварительную обработку текстов размеченного датасета

In [4]:
import re
import nltk
import pymorphy3
from nltk.corpus import stopwords

In [5]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dinislam/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/dinislam/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
morph = pymorphy3.MorphAnalyzer()

In [7]:
russian_stopwords = set(stopwords.words('russian'))
english_stopwords = set(stopwords.words('english'))
all_stopwords = russian_stopwords.union(english_stopwords)

создаём функции для обработки

In [8]:
def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ\s-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [9]:
def lemmatize_text(text, lang='russian'):
    words = text.split()
    lemmatized_words = []
    
    if lang == 'russian':
        for word in words:
            if word not in all_stopwords:
                lemmatized = morph.parse(word)[0].normal_form
                lemmatized_words.append(lemmatized)
    else:
        # если не русский
        from nltk.stem import WordNetLemmatizer
        nltk.download('wordnet')
        lemmatizer = WordNetLemmatizer()
        
        for word in words:
            if word not in all_stopwords:
                lemmatized = lemmatizer.lemmatize(word)
                lemmatized_words.append(lemmatized)
    
    return ' '.join(lemmatized_words)

In [10]:
def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in all_stopwords]
    return ' '.join(filtered_words)

## обработка данных

### очищение

In [11]:
df['cleaned_text'] = df['Text'].apply(clean_text)

### стоп слова

In [12]:
df['text_no_stopwords'] = df['cleaned_text'].apply(remove_stopwords)

### леметизация

In [13]:
import spacy

In [14]:
def lemmatize_multilingual(text, lang="english"):
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        return ""
    words = text.split()
    lemmatized_words = []

    if lang == "russian":
        for word in words:
            if word not in all_stopwords:
                lemmatized_words.append(morph.parse(word)[0].normal_form)
    else:
        from nltk.stem import WordNetLemmatizer
        lemmatizer = WordNetLemmatizer()
        for word in words:
            if word not in all_stopwords:
                lemmatized_words.append(lemmatizer.lemmatize(word))

    return " ".join(lemmatized_words)

In [15]:
df["cleanedText"] = df["Text"].apply(clean_text)
df["lemmatizedText"] = df.apply(
    lambda row: lemmatize_multilingual(row["cleanedText"], row["Language"].lower()),
    axis=1
)

## просмотр итогов

In [16]:
df.head(10)

,Text,Language,cleaned_text,text_no_stopwords,cleanedText,lemmatizedText
0,"Nature, in the broadest sense, is the natural...",English,nature in the broadest sense is the natural ph...,nature broadest sense natural physical materia...,nature in the broadest sense is the natural ph...,nature broadest sense natural physical materia...
1,"""Nature"" can refer to the phenomena of the phy...",English,nature can refer to the phenomena of the physi...,nature refer phenomena physical world also lif...,nature can refer to the phenomena of the physi...,nature refer phenomenon physical world also li...
2,"The study of nature is a large, if not the onl...",English,the study of nature is a large if not the only...,study nature large part science,the study of nature is a large if not the only...,study nature large part science
3,"Although humans are part of nature, human acti...",English,although humans are part of nature human activ...,although humans part nature human activity oft...,although humans are part of nature human activ...,although human part nature human activity ofte...
4,[1] The word nature is borrowed from the Old F...,English,the word nature is borrowed from the old frenc...,word nature borrowed old french nature derived...,the word nature is borrowed from the old frenc...,word nature borrowed old french nature derived...
5,"[2] In ancient philosophy, natura is mostly us...",English,in ancient philosophy natura is mostly used as...,ancient philosophy natura mostly used latin tr...,in ancient philosophy natura is mostly used as...,ancient philosophy natura mostly used latin tr...
6,"[3][4] \nThe concept of nature as a whole, the...",English,the concept of nature as a whole the physical ...,concept nature whole physical universe one sev...,the concept of nature as a whole the physical ...,concept nature whole physical universe one sev...
7,During the advent of modern scientific method ...,English,during the advent of modern scientific method ...,advent modern scientific method last several c...,during the advent of modern scientific method ...,advent modern scientific method last several c...
8,"[5][6] With the Industrial revolution, nature ...",English,with the industrial revolution nature increasi...,industrial revolution nature increasingly beca...,with the industrial revolution nature increasi...,industrial revolution nature increasingly beca...
9,"However, a vitalist vision of nature, closer t...",English,however a vitalist vision of nature closer to ...,however vitalist vision nature closer presocra...,however a vitalist vision of nature closer to ...,however vitalist vision nature closer presocra...


## Сохранение результатов обработочки данных

In [17]:
df.to_csv('TextLanguage_nonoyesyes.csv', index=False)

# Провести тематическое моделирование

In [18]:
from wordcloud import WordCloud
from gensim import corpora, models
from gensim.models import LdaModel, CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt

#### для русских шрифтов

In [19]:
plt.rcParams['axes.unicode_minus'] = False

## облако

In [20]:
print(df.columns.tolist())

['Text', 'Language', 'cleaned_text', 'text_no_stopwords', 'cleanedText', 'lemmatizedText']


In [21]:
text_for_cloud = ' '.join(df['lemmatizedText'].dropna().astype(str))

In [22]:
def create_wordcloud(text, title):
    wordcloud = WordCloud(width=800, height=400, 
                         background_color='white',
                         max_words=100,
                         colormap='viridis',
                         random_state=42).generate(text)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Облако слов - {title}', fontsize=16, pad=20)
    plt.tight_layout()
    plt.show()

In [ ]:
languages = df['Language'].unique()
print("язык:", languages)

for lang in df['Language'].unique():
    lang_texts = df[df['Language'] == lang]['lemmatizedText']
    
    if len(lang_texts) > 0:
        combined_text = ' '.join(lang_texts.dropna().astype(str))
        
        create_wordcloud(combined_text, title=f'Лем тексты - {lang}')

## подготовочка для тематического моделирования

In [24]:
def prepare_corpus(text_series):
    # тексты на слова
    tokenized_texts = [text.split() for text in text_series if isinstance(text, str) and text.strip()]
    
    # словарь
    dictionary = corpora.Dictionary(tokenized_texts)
    
    # редкие и частые слова
    dictionary.filter_extremes(no_below=2, no_above=0.5)
    
    # корпус в формате рюкзак слов
    corpus = [dictionary.doc2bow(text) for text in tokenized_texts]
    
    return tokenized_texts, dictionary, corpus

In [25]:
## лда моделирования для каждого язка

In [26]:
lda_models = {}
topics_data = {}
dictionaries = {}
corpora_dict = {}

for lang in languages:
    print(f"\n=== Тематическое моделирование для языка: {lang} ===")
    
    lang_texts = df[df['Language'] == lang]['lemmatizedText'].dropna()
    
    if len(lang_texts) < 5:
        print(f"Недостаточно текстов для {lang} (только {len(lang_texts)})")
        continue
    
    tokenized_texts, dictionary, corpus = prepare_corpus(lang_texts)
    
    dictionaries[lang] = dictionary
    corpora_dict[lang] = corpus
    
    lda_model = LdaModel(corpus=corpus,
                        id2word=dictionary,
                        num_topics=5,
                        random_state=42,
                        passes=10)
    
    lda_models[lang] = lda_model
    topics = lda_model.print_topics(num_words=10)
    topics_data[lang] = topics
    
    print(f"\nТемы для {lang}:")
    for topic_id, topic in topics:
        print(f"\nТема {topic_id + 1}:")
        words = [word.split('*')[1].strip('"') for word in topic.split(' + ')]
        for word in words[:10]:
            print(f"  {word}")


=== Тематическое моделирование для языка: English ===

Темы для English:

Тема 1:
  learning
  data
  know
  earth
  time
  example
  system
  feature
  word
  sorry

Тема 2:
  learning
  data
  machine
  wikipedia
  nature
  artificial
  training
  set
  algorithm
  human

Тема 3:
  wikipedia
  content
  page
  article
  project
  language
  also
  word
  one
  life

Тема 4:
  learning
  wikipedia
  machine
  million
  also
  word
  life
  one
  wikimedia
  article

Тема 5:
  wikipedia
  article
  editor
  time
  say
  one
  learning
  would
  good
  may

=== Тематическое моделирование для языка: Malayalam ===

Темы для Malayalam:

Тема 1:
  nature
  loss
  -
  machine
  learning
  error
  squared
  mean

Тема 2:
  squared
  mean
  error
  loss
  nature
  -
  machine
  learning

Тема 3:
  -
  loss
  learning
  machine
  nature
  squared
  mean
  error

Тема 4:
  -
  loss
  nature
  learning
  machine
  error
  mean
  squared

Тема 5:
  -
  learning
  machine
  nature
  loss
  mean
  

In [27]:
import numpy as np

In [28]:
for lang, lda_model in lda_models.items():
    if lda_model:
        print(f"\nВизуализация тем для языка {lang}")
        try:
            # Получаем словарь и корпус для этого языка
            lang_dict = dictionaries[lang]
            lang_corpus = corpora_dict[lang]
                     # ваш корпус для языка
            
            # Проверяем нормализацию topic_term_dists
            topic_dists = lda_model.get_topics()  # shape: (n_topics, n_terms)
            row_sums = topic_dists.sum(axis=1)
            
            if not np.allclose(row_sums, 1.0, atol=1e-6):
                print(f"  Внимание: суммы строк = {row_sums[:3]}, нормализуем...")
                # Нормализуем вручную
                lda_model.state.sstats = lda_model.state.sstats / lda_model.state.sstats.sum(axis=1)[:, np.newaxis]
            
            # Создаем визуализацию
            vis_data = pyLDAvis.gensim_models.prepare(lda_model, 
                                                      lang_corpus, 
                                                      lang_dict,
                                                      mds='tsne')
            pyLDAvis.save_html(vis_data, f'lda_visualization_{lang}.html')
            print(f"  ✓ Визуализация сохранена в файл: lda_visualization_{lang}.html")
            
        except Exception as e:
            print(f"  ✗ Не удалось создать визуализацию: {e}")


Визуализация тем для языка English
  ✓ Визуализация сохранена в файл: lda_visualization_English.html

Визуализация тем для языка Malayalam
  ✓ Визуализация сохранена в файл: lda_visualization_Malayalam.html

Визуализация тем для языка Hindi
  ✓ Визуализация сохранена в файл: lda_visualization_Hindi.html

Визуализация тем для языка Tamil
  ✓ Визуализация сохранена в файл: lda_visualization_Tamil.html

Визуализация тем для языка Portugeese
  ✓ Визуализация сохранена в файл: lda_visualization_Portugeese.html

Визуализация тем для языка French
  ✓ Визуализация сохранена в файл: lda_visualization_French.html

Визуализация тем для языка Dutch
  ✓ Визуализация сохранена в файл: lda_visualization_Dutch.html

Визуализация тем для языка Spanish
  ✓ Визуализация сохранена в файл: lda_visualization_Spanish.html

Визуализация тем для языка Greek
  ✓ Визуализация сохранена в файл: lda_visualization_Greek.html

Визуализация тем для языка Russian
  ✓ Визуализация сохранена в файл: lda_visualization_R

## специфичные слова

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

for lang in languages:
    print(f"\n{lang}")
    
    lang_texts = df[df['Language'] == lang]['lemmatizedText'].dropna()
    other_texts = df[df['Language'] != lang]['lemmatizedText'].dropna()
    
    if len(lang_texts) < 2:
        print(f"Недостаточно текстов для анализа")
        continue
    
    all_texts = list(lang_texts) + list(other_texts) if len(other_texts) > 0 else list(lang_texts)
    
    try:
        vectorizer = TfidfVectorizer(max_features=50, token_pattern=r'\b[a-zA-Zа-яА-Я]+\b')
        tfidf_matrix = vectorizer.fit_transform(all_texts)
        
        lang_tfidf = tfidf_matrix[:len(lang_texts)].mean(axis=0).A1
        other_tfidf = tfidf_matrix[len(lang_texts):].mean(axis=0).A1 if len(other_texts) > 0 else lang_tfidf * 0
        
        # разница
        feature_names = vectorizer.get_feature_names_out()
        diff_scores = lang_tfidf - other_tfidf
        
        # топ-10 специфичных слов
        top_indices = diff_scores.argsort()[-10:][::-1]
        
        print("Топ-10 специфических слов:")
        for idx in top_indices:
            if diff_scores[idx] > 0:
                print(f"  {feature_names[idx]}: {diff_scores[idx]:.4f}")
                
    except Exception as e:
        print(f"Ошибка при анализе TF-IDF: {e}")


English
Топ-10 специфических слов:
  wikipedia: 0.1819
  nature: 0.0189

Malayalam
Топ-10 специфических слов:

Hindi
Топ-10 специфических слов:
  w: 0.0111
  dia: 0.0053
  h: 0.0015

Tamil
Топ-10 специфических слов:

Portugeese
Топ-10 специфических слов:
  e: 0.1576
  que: 0.1431
  de: 0.1429
  em: 0.1411
  voc: 0.1376
  dia: 0.1143
  um: 0.1038
  wikip: 0.0908
  para: 0.0863
  da: 0.0861

French
Топ-10 специфических слов:
  de: 0.2220
  le: 0.2141
  et: 0.1288
  la: 0.1229
  l: 0.1182
  vous: 0.1041
  est: 0.0975
  une: 0.0912
  un: 0.0738
  en: 0.0527

Dutch
Топ-10 специфических слов:
  het: 0.2544
  je: 0.2478
  en: 0.1109
  de: 0.0865
  te: 0.0800
  al: 0.0709
  om: 0.0471
  er: 0.0346
  kan: 0.0297
  wikipedia: 0.0195

Spanish
Топ-10 специфических слов:
  de: 0.1912
  la: 0.1687
  que: 0.1561
  el: 0.1446
  en: 0.1245
  n: 0.0998
  por: 0.0989
  una: 0.0689
  un: 0.0677
  se: 0.0594

Greek
Топ-10 специфических слов:

Russian
Топ-10 специфических слов:
  nature: 0.0053

Danish
Топ

## Частота слов

In [ ]:
for lang in languages:
    print(f"\n{lang.upper()} язык:")
    print("-" * 40)
    
    lang_texts = df[df['Language'] == lang]['lemmatizedText'].dropna()
    all_words = ' '.join(lang_texts).split()
    
    word_freq = Counter(all_words)
    
    print("Топ-20:")
    for i, (word, freq) in enumerate(word_freq.most_common(20), 1):
        print(f"  {i:2d}. {word}: {freq}")
    
    top_words = word_freq.most_common(15)
    words, freqs = zip(*top_words)
    
    plt.figure(figsize=(12, 6))
    plt.bar(words, freqs, color='skyblue', alpha=0.7)
    plt.title(f'Топ-15 частотных слов - {lang}', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Векторизация. Получить векторное представление текстов

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [32]:
texts = df['lemmatizedText'].fillna('').astype(str)

In [33]:
vectorizer = TfidfVectorizer(
    max_features=1000,
    min_df=1,
    max_df=0.9,
    token_pattern=r'(?u)\b\w+\b',
    # ngram_range=(1, 2)
)

In [34]:
tfidf_matrix = vectorizer.fit_transform(texts)

In [35]:
features = vectorizer.get_feature_names_out()

In [36]:
tfidf_matrix.shape

(10337, 1000)

In [37]:
features[:20]

array(['aan', 'aber', 'ad', 'af', 'ai', 'ainsi', 'ak', 'al', 'alem',
       'algo', 'algorithm', 'algu', 'alguien', 'algunas', 'algunos',
       'alguns', 'alla', 'alle', 'alles', 'alors'], dtype=object)

In [38]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=features,
    index=df.index
)

In [39]:
tfidf_df['Language'] = df['Language'].values


In [40]:
print(tfidf_df.head())

   aan  aber   ad   af   ai  ainsi   ak   al  alem  algo  ...  участник  \
0  0.0   0.0  0.0  0.0  0.0    0.0  0.0  0.0   0.0   0.0  ...       0.0   
1  0.0   0.0  0.0  0.0  0.0    0.0  0.0  0.0   0.0   0.0  ...       0.0   
2  0.0   0.0  0.0  0.0  0.0    0.0  0.0  0.0   0.0   0.0  ...       0.0   
3  0.0   0.0  0.0  0.0  0.0    0.0  0.0  0.0   0.0   0.0  ...       0.0   
4  0.0   0.0  0.0  0.0  0.0    0.0  0.0  0.0   0.0   0.0  ...       0.0   

   фраза  хороший  хотеть  человек  что  это  этот  являться  Language  
0    0.0      0.0     0.0      0.0  0.0  0.0   0.0       0.0   English  
1    0.0      0.0     0.0      0.0  0.0  0.0   0.0       0.0   English  
2    0.0      0.0     0.0      0.0  0.0  0.0   0.0       0.0   English  
3    0.0      0.0     0.0      0.0  0.0  0.0   0.0       0.0   English  
4    0.0      0.0     0.0      0.0  0.0  0.0   0.0       0.0   English  

[5 rows x 1001 columns]


In [41]:
# Давайте проверим, насколько разреженная матрица
sparsity = 1.0 - (tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]))
print(f"Разреженность: {sparsity:.2%}")
# Скорее всего покажет > 99% - почти все нули

# Проверим ненулевые значения на один документ
print(f"Ненулевых признаков в среднем: {tfidf_matrix.nnz / tfidf_matrix.shape[0]:.1f}")

Разреженность: 99.42%
Ненулевых признаков в среднем: 5.8


In [42]:
# Проверьте длину текстов до и после обработки
print("До обработки (оригинал):")
print(f"  Средняя длина: {df['Text'].str.len().mean():.1f} символов")
print(f"  Пример: {df['Text'].iloc[0][:200]}")

print("\nПосле очистки (cleanedText):")
print(f"  Средняя длина: {df['cleanedText'].str.len().mean():.1f} символов")
print(f"  Пример: {df['cleanedText'].iloc[0][:200]}")

print("\nПосле лемматизации (lemmatizedText):")
print(f"  Средняя длина: {df['lemmatizedText'].str.len().mean():.1f} символов")
print(f"  Пример: {df['lemmatizedText'].iloc[0][:200]}")

print("\nСреднее количество слов в лемматизированном тексте:")
print(f"  {df['lemmatizedText'].str.split().str.len().mean():.1f} слов")

До обработки (оригинал):
  Средняя длина: 124.1 символов
  Пример:  Nature, in the broadest sense, is the natural, physical, material world or universe.

После очистки (cleanedText):
  Средняя длина: 89.9 символов
  Пример: nature in the broadest sense is the natural physical material world or universe

После лемматизации (lemmatizedText):
  Средняя длина: 80.6 символов
  Пример: nature broadest sense natural physical material world universe

Среднее количество слов в лемматизированном тексте:
  13.4 слов
